# Run-of-River production calculation: Capped and Uncapped

This notebook includes the minimal setup that run the ParFlow alluvium workflow capped and uncapped, then compare the results.

In [1]:
# Minimal imports and repo root detection
import pandas as pd
import numpy as np
from pathlib import Path

rng = np.random.default_rng(42)
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

In [2]:
# Plant placement and time index (same as original notebook)

hydro_name = ["Bergheim", "Vohburg"]
hydro_id = ["GHR03171", "GHR03272"]
hydro_lat = [48.750807, 48.778103]
hydro_lon = [11.273237, 11.60119]
hydro_capacity = [23_700.0, 23_300.0]  # capacity in kW
hydro_head = [6, 6.07]  # m
placements = pd.DataFrame(
    {
        "lat": hydro_lat,
        "lon": hydro_lon,
        "capacity": hydro_capacity,  # capacity in kW
    }
)
year = 2020
time_index = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
net_head_m = hydro_head
placements

,lat,lon,capacity
0,48.750807,11.273237,23700.0
1,48.778103,11.601190,23300.0


In [3]:
# Paths to ParFlow extraction directory and auxiliary files (adjust if unavailable)
root_dir = repo_root / "reskit" / "hydro" / "_external_module" / "parflow_600m_runs"
alluvium_mask_file = root_dir / "alluvium_mask.nc"
indicator_file = root_dir / "DE-0055_INDICATOR_regridded_rescaled_SoilGrids250-v2017_BGRvector_newAllv.nc"
root_dir, alluvium_mask_file, indicator_file

(PosixPath('/fast/home/s-chen/local_bin/python_envs_managed_by_mamba/reskit_hydropower/reskit/reskit/hydro/_external_module/parflow_600m_runs'),
 PosixPath('/fast/home/s-chen/local_bin/python_envs_managed_by_mamba/reskit_hydropower/reskit/reskit/hydro/_external_module/parflow_600m_runs/alluvium_mask.nc'),
 PosixPath('/fast/home/s-chen/local_bin/python_envs_managed_by_mamba/reskit_hydropower/reskit/reskit/hydro/_external_module/parflow_600m_runs/DE-0055_INDICATOR_regridded_rescaled_SoilGrids250-v2017_BGRvector_newAllv.nc'))

In [ ]:
# cap production by capacity (ParFlow-based extraction)
from reskit.hydro.workflows import run_of_river_parflow_alluvium_workflow

full_parflow_result_capped = run_of_river_parflow_alluvium_workflow(
    placements=placements,
    year=year,
    time_index=time_index,
    net_head_m=net_head_m,
    extraction_root_dir=str(root_dir),
    alluvium_mask_file=str(alluvium_mask_file),
    indicator_file=str(indicator_file),
    efficiency=0.88,
    cap_production_by_capacity=True,
    output_variables=[
        "discharge_m3_per_day",
        "usable_discharge_m3_per_day",
        "capacity_factor",
        "total_system_generation",
    ],
)

# "discharge_m3_per_day": available discharge in m³/day (not capped by capacity)
# "usable_discharge_m3_per_day": discharge in m³/day that can be used for power generation (capped by capacity)
# with cap_production_by_capacity=True, the usable discharge is capped by the plant's capacity, while the available discharge is not capped and represents the actual water flow at the location. The capacity factor and total system generation are calculated based on the usable discharge.

full_parflow_result_capped[
    ["discharge_m3_per_day", "usable_discharge_m3_per_day", "capacity_factor", "total_system_generation"]
]

/fast/home/s-chen/local_bin/python_envs_managed_by_mamba/reskit_hydropower/reskit/reskit/hydro/_external_module/parflow_600m_runs/parflow_data_extraction.py:138: RuntimeWarning: invalid value encountered in arccos
  spherical_distance = Rearth * np.arccos(np.sin(lat1) * np.sin(lat2) + np.cos(lat1) * np.cos(lat2) * np.cos(lon2 - lon1))


----------------------------------------
JSON record: 0
location ID, lon, lat, depth, sim-data: location_0, 11.275018692016602, 48.747344970703125, 0.2, https://service.tereno.net/thredds/dodsC/forecastnrw/products/ParFlow-DE06-HC_v03/sfd_DE05_ECMWF-HRES_hindcast_r1i1p2_FZJ-IBG3-ParFlowCLM380_hgfadapter-h00-v03bJuwelsGpuProdClimatologyTl_1day_20200101-20201231.nc
POI: index x-y and lon-lat on model grid: 1163, 652, 11.275018692016602, 48.747344970703125
ATTENTION: your chosen POI (point of interest) is located directly on a lake, river, or ocean grid element, it is recommended to specify an alternative, nearby longitude-latitude coordinate pair
indices of alternative close-by grid elements to POI:  [(652, 1164), (652, 1162), (651, 1163), (653, 1163), (651, 1162), (651, 1164), (653, 1162), (653, 1164), (652, 1165)]
alternative, recommended coordinate pairs close-by, not on river, lake, or ocean, Lon-Lat [dec deg], 5 digits, edit your JASON file accordingly:  11.26672 48.74685
alternativ

<xarray.Dataset> Size: 26kB
Dimensions:                      (time: 366, location: 2)
Coordinates:
  * time                         (time) datetime64[ns] 3kB 2020-01-01 ... 202...
  * location                     (location) int64 16B 0 1
Data variables:
    discharge_m3_per_day         (time, location) float64 6kB 3.64e+07 ... 3....
    usable_discharge_m3_per_day  (time, location) float64 6kB 3.64e+07 ... 3....
    capacity_factor              (time, location) float64 6kB 0.9206 ... 0.9612
    total_system_generation      (time, location) float64 6kB 5.237e+05 ... 5...

In [ ]:
# no cap on production (ParFlow-based extraction)
from reskit.hydro.workflows import run_of_river_parflow_alluvium_workflow

full_parflow_result_uncapped = run_of_river_parflow_alluvium_workflow(
    placements=placements,
    year=year,
    time_index=time_index,
    net_head_m=net_head_m,
    extraction_root_dir=str(root_dir),
    alluvium_mask_file=str(alluvium_mask_file),
    indicator_file=str(indicator_file),
    efficiency=0.88,
    cap_production_by_capacity=False,
    output_variables=[
        "discharge_m3_per_day",
        "usable_discharge_m3_per_day",
        "capacity_factor",
        "total_system_generation",
    ],
)

# with cap_production_by_capacity=False, the usable discharge is not capped and represents the actual water flow at the location. That being said, "discharge_m3_per_day" = "usable_discharge_m3_per_day"
# The capacity factor and total system generation are calculated based on the uncapped usable discharge.

full_parflow_result_uncapped[
    ["discharge_m3_per_day", "usable_discharge_m3_per_day", "capacity_factor", "total_system_generation"]
]

/fast/home/s-chen/local_bin/python_envs_managed_by_mamba/reskit_hydropower/reskit/reskit/hydro/_external_module/parflow_600m_runs/parflow_data_extraction.py:138: RuntimeWarning: invalid value encountered in arccos
  spherical_distance = Rearth * np.arccos(np.sin(lat1) * np.sin(lat2) + np.cos(lat1) * np.cos(lat2) * np.cos(lon2 - lon1))


----------------------------------------
JSON record: 0
location ID, lon, lat, depth, sim-data: location_0, 11.275018692016602, 48.747344970703125, 0.2, https://service.tereno.net/thredds/dodsC/forecastnrw/products/ParFlow-DE06-HC_v03/sfd_DE05_ECMWF-HRES_hindcast_r1i1p2_FZJ-IBG3-ParFlowCLM380_hgfadapter-h00-v03bJuwelsGpuProdClimatologyTl_1day_20200101-20201231.nc
POI: index x-y and lon-lat on model grid: 1163, 652, 11.275018692016602, 48.747344970703125
ATTENTION: your chosen POI (point of interest) is located directly on a lake, river, or ocean grid element, it is recommended to specify an alternative, nearby longitude-latitude coordinate pair
indices of alternative close-by grid elements to POI:  [(652, 1164), (652, 1162), (651, 1163), (653, 1163), (651, 1162), (651, 1164), (653, 1162), (653, 1164), (652, 1165)]
alternative, recommended coordinate pairs close-by, not on river, lake, or ocean, Lon-Lat [dec deg], 5 digits, edit your JASON file accordingly:  11.26672 48.74685
alternativ

<xarray.Dataset> Size: 26kB
Dimensions:                      (time: 366, location: 2)
Coordinates:
  * time                         (time) datetime64[ns] 3kB 2020-01-01 ... 202...
  * location                     (location) int64 16B 0 1
Data variables:
    discharge_m3_per_day         (time, location) float64 6kB 3.64e+07 ... 3....
    usable_discharge_m3_per_day  (time, location) float64 6kB 3.64e+07 ... 3....
    capacity_factor              (time, location) float64 6kB 0.9206 ... 0.9612
    total_system_generation      (time, location) float64 6kB 5.237e+05 ... 5...

In [6]:
# Compare capped vs uncapped ParFlow-based run-of-river outputs
capped_result = full_parflow_result_capped
uncapped_result = full_parflow_result_uncapped

# Compute differences (uncapped - capped)
diff_generation = uncapped_result["total_system_generation"] - capped_result["total_system_generation"]
diff_cf = uncapped_result["capacity_factor"] - capped_result["capacity_factor"]

print("Summary: total_system_generation (uncapped - capped)")
print(diff_generation.to_dataframe().describe())
print("\nSummary: capacity_factor (uncapped - capped)")
print(diff_cf.to_dataframe().describe())

# Return the two difference arrays for inspection
diff_generation, diff_cf

Summary: total_system_generation (uncapped - capped)
       total_system_generation
count             7.320000e+02
mean              4.859799e+04
std               1.819550e+05
min               0.000000e+00
25%               0.000000e+00
50%               0.000000e+00
75%               0.000000e+00
max               1.548383e+06

Summary: capacity_factor (uncapped - capped)
       capacity_factor
count       732.000000
mean          0.086246
std           0.322848
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           2.768926


(<xarray.DataArray 'total_system_generation' (time: 366, location: 2)> Size: 6kB
 array([[0.00000000e+00, 2.31531758e+04],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00],
 ...
        [0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000